# 03_feature_engineering.ipynb

Converted from your cell-by-cell rebuild guide.

## Cell 1 — Imports

In [1]:
import os
import pandas as pd
import numpy as np
from ta import trend, momentum, volatility, volume as ta_volume
import warnings
warnings.filterwarnings("ignore")

# Load cleaned price data from Notebook 1
price = pd.read_csv("price_clean.csv", parse_dates=["date"])

# Load Bitcoin-only daily Twitter features from Notebook 1
btc_twitter_path = "bitcoin_twitter_daily.csv"
expected_twitter_cols = [
    "coin",
    "date",
    "btc_twitter_sentiment_mean",
    "btc_twitter_sentiment_std",
    "btc_twitter_tweet_count",
]

if os.path.exists(btc_twitter_path):
    twitter_btc_daily = pd.read_csv(btc_twitter_path, parse_dates=["date"])
else:
    twitter_btc_daily = pd.DataFrame(columns=expected_twitter_cols)

# Ensure required columns exist even if file is empty / stale
for col in expected_twitter_cols:
    if col not in twitter_btc_daily.columns:
        twitter_btc_daily[col] = np.nan

twitter_btc_daily = twitter_btc_daily[expected_twitter_cols].copy()

COINS = ["Bitcoin", "Ethereum", "Dogecoin"]

print("price shape:", price.shape)
print("bitcoin_twitter_daily shape:", twitter_btc_daily.shape)
print("bitcoin_twitter_daily columns:", twitter_btc_daily.columns.tolist())


price shape: (7594, 8)
bitcoin_twitter_daily shape: (2754, 5)
bitcoin_twitter_daily columns: ['coin', 'date', 'btc_twitter_sentiment_mean', 'btc_twitter_sentiment_std', 'btc_twitter_tweet_count']


## Cell 2 — Build per-coin technical indicators (coin-agnostic column names)

In [2]:
def build_technical_features(df_coin: pd.DataFrame) -> pd.DataFrame:
    """
    Input: single-coin dataframe with columns: date, open, high, low, close, volume
    Output: same df + technical indicator columns — ALL coin-agnostic names
    """
    df = df_coin.sort_values("date").copy()

    # ── Returns & lags ────────────────────────────────────────────────────
    df["return_1"]  = df["close"].pct_change(1)
    df["return_3"]  = df["close"].pct_change(3)
    df["return_7"]  = df["close"].pct_change(7)
    df["return_14"] = df["close"].pct_change(14)
    df["return_30"] = df["close"].pct_change(30)

    # ── Rolling statistics ────────────────────────────────────────────────
    for w in [7, 14, 30]:
        df[f"rolling_mean_{w}"]  = df["close"].rolling(w).mean()
        df[f"rolling_std_{w}"]   = df["close"].rolling(w).std()
        df[f"rolling_min_{w}"]   = df["close"].rolling(w).min()
        df[f"rolling_max_{w}"]   = df["close"].rolling(w).max()

    # Price relative to rolling mean (normalised level)
    df["price_vs_ma7"]  = df["close"] / df["rolling_mean_7"]  - 1
    df["price_vs_ma14"] = df["close"] / df["rolling_mean_14"] - 1
    df["price_vs_ma30"] = df["close"] / df["rolling_mean_30"] - 1

    # ── Volatility features ───────────────────────────────────────────────
    df["range"]          = (df["high"] - df["low"]) / df["close"]   # daily range %
    df["upper_shadow"]   = (df["high"] - df[["open","close"]].max(axis=1)) / df["close"]
    df["lower_shadow"]   = (df[["open","close"]].min(axis=1) - df["low"]) / df["close"]
    df["body_size"]      = abs(df["close"] - df["open"]) / df["close"]
    df["atr_14"]         = volatility.AverageTrueRange(
                               df["high"], df["low"], df["close"], window=14
                           ).average_true_range()

    # ── Momentum & trend indicators ───────────────────────────────────────
    df["rsi_14"]  = momentum.RSIIndicator(df["close"], window=14).rsi()
    df["rsi_7"]   = momentum.RSIIndicator(df["close"], window=7).rsi()

    macd_obj     = trend.MACD(df["close"])
    df["macd"]        = macd_obj.macd()
    df["macd_signal"] = macd_obj.macd_signal()
    df["macd_diff"]   = macd_obj.macd_diff()

    df["adx_14"]  = trend.ADXIndicator(df["high"], df["low"], df["close"], window=14).adx()
    df["cci_20"]  = trend.CCIIndicator(df["high"], df["low"], df["close"], window=20).cci()

    bb = volatility.BollingerBands(df["close"], window=20)
    df["bb_upper"]  = bb.bollinger_hband()
    df["bb_lower"]  = bb.bollinger_lband()
    df["bb_width"]  = bb.bollinger_wband()
    df["bb_pct"]    = bb.bollinger_pband()       # position within bands

    # ── Volume features ───────────────────────────────────────────────────
    df["volume_change"]   = df["volume"].pct_change(1)
    df["volume_ma7"]      = df["volume"].rolling(7).mean()
    df["volume_vs_ma7"]   = df["volume"] / df["volume_ma7"] - 1

    obv = ta_volume.OnBalanceVolumeIndicator(df["close"], df["volume"])
    df["obv"] = obv.on_balance_volume()
    df["obv_change"] = df["obv"].pct_change(1)

    # ── Calendar features ─────────────────────────────────────────────────
    df["dayofweek"]  = df["date"].dt.dayofweek
    df["month"]      = df["date"].dt.month
    df["quarter"]    = df["date"].dt.quarter
    df["is_month_end"]   = df["date"].dt.is_month_end.astype(int)
    df["is_month_start"] = df["date"].dt.is_month_start.astype(int)

    # ── Lag features (yesterday's direction) ─────────────────────────────
    df["lag1_return"] = df["return_1"].shift(1)
    df["lag2_return"] = df["return_1"].shift(2)
    df["lag3_return"] = df["return_1"].shift(3)

    return df

# Apply per coin
all_coins = []
for coin in COINS:
    g = price[price["coin"]==coin].copy()
    g = build_technical_features(g)
    # Target: next day up = 1, down = 0
    g["target"] = (g["close"].shift(-1) > g["close"]).astype("Int64")
    all_coins.append(g)

price_feat = pd.concat(all_coins, ignore_index=True)
print(price_feat.shape)
price_feat.head(3)

(7594, 58)


,coin,date,open,high,low,close,volume,pct_change_raw,return_1,return_3,...,obv_change,dayofweek,month,quarter,is_month_end,is_month_start,lag1_return,lag2_return,lag3_return,target
0,Bitcoin,2010-07-18,0.0,0.1,0.1,0.1,80.0,0.0,NaN,NaN,...,NaN,6,7,3,0,0,NaN,NaN,NaN,0
1,Bitcoin,2010-07-19,0.1,0.1,0.1,0.1,570.0,0.0,0.0,NaN,...,7.125,0,7,3,0,0,NaN,NaN,NaN,0
2,Bitcoin,2010-07-20,0.1,0.1,0.1,0.1,260.0,0.0,0.0,NaN,...,0.400,1,7,3,0,0,0.0,NaN,NaN,0


## Cell 3 — Merge Twitter sentiment (Bitcoin-only enhancement)

In [3]:
# Use Bitcoin-only daily Twitter features produced in Notebook 1
twitter_daily = twitter_btc_daily.copy()

# Make sure date is parsed correctly
twitter_daily["date"] = pd.to_datetime(twitter_daily["date"], errors="coerce")

# Keep only valid Bitcoin rows
twitter_daily = twitter_daily[
    (twitter_daily["coin"] == "Bitcoin") & twitter_daily["date"].notna()
].copy()

if len(twitter_daily) == 0:
    print("No Bitcoin Twitter rows found in bitcoin_twitter_daily.csv.")
    print("Proceeding with price-only features, but keeping BTC Twitter columns as zeros.")

    twitter_daily = pd.DataFrame({
        "coin": pd.Series(dtype="object"),
        "date": pd.Series(dtype="datetime64[ns]"),
        "btc_twitter_sentiment_mean": pd.Series(dtype="float"),
        "btc_twitter_sentiment_std": pd.Series(dtype="float"),
        "btc_twitter_tweet_count": pd.Series(dtype="float"),
        "btc_twitter_sentiment_ma3": pd.Series(dtype="float"),
        "btc_twitter_sentiment_ma7": pd.Series(dtype="float"),
        "btc_twitter_sentiment_chg3": pd.Series(dtype="float"),
        "btc_twitter_sentiment_chg7": pd.Series(dtype="float"),
        "btc_twitter_tweet_count_ma3": pd.Series(dtype="float"),
        "btc_twitter_tweet_count_ma7": pd.Series(dtype="float"),
    })

else:
    twitter_daily = twitter_daily.sort_values(["coin", "date"]).copy()

    # Rolling Bitcoin Twitter features
    for w in [3, 7]:
        twitter_daily[f"btc_twitter_sentiment_ma{w}"] = (
            twitter_daily.groupby("coin")["btc_twitter_sentiment_mean"]
            .transform(lambda x: x.rolling(w, min_periods=1).mean())
        )

        twitter_daily[f"btc_twitter_sentiment_chg{w}"] = (
            twitter_daily.groupby("coin")["btc_twitter_sentiment_mean"]
            .transform(lambda x: x.pct_change(w))
        )

        twitter_daily[f"btc_twitter_tweet_count_ma{w}"] = (
            twitter_daily.groupby("coin")["btc_twitter_tweet_count"]
            .transform(lambda x: x.rolling(w, min_periods=1).mean())
        )

# Merge into the engineered price features
merged = pd.merge(price_feat, twitter_daily, on=["coin", "date"], how="left")

# Fill Bitcoin Twitter features with 0 when unavailable
twitter_feature_base_cols = [
    "btc_twitter_sentiment_mean",
    "btc_twitter_sentiment_std",
    "btc_twitter_tweet_count",
    "btc_twitter_sentiment_ma3",
    "btc_twitter_sentiment_ma7",
    "btc_twitter_sentiment_chg3",
    "btc_twitter_sentiment_chg7",
    "btc_twitter_tweet_count_ma3",
    "btc_twitter_tweet_count_ma7",
]

for col in twitter_feature_base_cols:
    if col not in merged.columns:
        merged[col] = 0.0

merged[twitter_feature_base_cols] = (
    merged[twitter_feature_base_cols]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("After merge:", merged.shape)
print("BTC Twitter feature columns:", twitter_feature_base_cols)

btc_nonzero_days = int((merged.loc[merged["coin"] == "Bitcoin", "btc_twitter_tweet_count"] > 0).sum())
print("Bitcoin rows with non-zero Twitter coverage after merge:", btc_nonzero_days)

print("Twitter feature design: Bitcoin uses sentiment; Ethereum and Dogecoin remain price-only at export.")


After merge: (7594, 67)
BTC Twitter feature columns: ['btc_twitter_sentiment_mean', 'btc_twitter_sentiment_std', 'btc_twitter_tweet_count', 'btc_twitter_sentiment_ma3', 'btc_twitter_sentiment_ma7', 'btc_twitter_sentiment_chg3', 'btc_twitter_sentiment_chg7', 'btc_twitter_tweet_count_ma3', 'btc_twitter_tweet_count_ma7']
Bitcoin rows with non-zero Twitter coverage after merge: 2754
Twitter feature design: Bitcoin uses sentiment; Ethereum and Dogecoin remain price-only at export.


## Cell 4 — Feature selection: drop low-variance & near-duplicate columns

In [4]:
from sklearn.feature_selection import VarianceThreshold

# Identify ALL numeric feature columns (exclude identifiers, raw OHLC, target)
exclude = ["coin", "date", "open", "high", "low", "close", "volume", "pct_change_raw", "target"]
feat_cols = [c for c in merged.columns if c not in exclude]

# Clean numeric matrix for variance filtering
X_all = merged[feat_cols].apply(pd.to_numeric, errors="coerce")

# Replace +/-inf created by pct_change or divide-by-zero style features
inf_mask = np.isinf(X_all.to_numpy())
inf_count = int(inf_mask.sum())
if inf_count > 0:
    inf_cols = X_all.columns[np.isinf(X_all).any(axis=0)].tolist()
    print(f"Replacing {inf_count} infinite values before variance filtering.")
    print("Columns containing inf:", inf_cols)

X_all = X_all.replace([np.inf, -np.inf], np.nan).fillna(0)

selector = VarianceThreshold(threshold=1e-5)
selector.fit(X_all)
kept = [c for c, s in zip(feat_cols, selector.get_support()) if s]
print(f"Features before variance filter: {len(feat_cols)}")
print(f"Features after variance filter:  {len(kept)}")
feat_cols = kept


Replacing 6 infinite values before variance filtering.
Columns containing inf: ['volume_change', 'obv_change']
Features before variance filter: 58
Features after variance filter:  58


## Cell 5 — Encoding cyclic calendar features

In [5]:
# Encode dayofweek and month as sine/cosine pairs (cyclic)
merged["dow_sin"] = np.sin(2 * np.pi * merged["dayofweek"] / 7)
merged["dow_cos"] = np.cos(2 * np.pi * merged["dayofweek"] / 7)
merged["month_sin"] = np.sin(2 * np.pi * merged["month"] / 12)
merged["month_cos"] = np.cos(2 * np.pi * merged["month"] / 12)

# Remove raw dayofweek / month (replaced by cyclic encoding)
feat_cols = [c for c in feat_cols if c not in ["dayofweek", "month"]]
feat_cols += ["dow_sin", "dow_cos", "month_sin", "month_cos"]
feat_cols = list(dict.fromkeys(feat_cols))

twitter_feature_cols = [c for c in feat_cols if "sentiment" in c or "tweet_count" in c]
print("Final feature count (master list):", len(feat_cols))
print("Twitter feature columns:", twitter_feature_cols)


Final feature count (master list): 60
Twitter feature columns: ['btc_twitter_sentiment_mean', 'btc_twitter_sentiment_std', 'btc_twitter_tweet_count', 'btc_twitter_sentiment_ma3', 'btc_twitter_sentiment_chg3', 'btc_twitter_tweet_count_ma3', 'btc_twitter_sentiment_ma7', 'btc_twitter_sentiment_chg7', 'btc_twitter_tweet_count_ma7']


## Cell 6 — Save per-coin feature datasets (Bitcoin gets Twitter; ETH/DOGE price-only)

In [6]:
feature_cols_by_coin = {}

for coin in COINS:
    g = merged[merged["coin"] == coin].dropna(subset=["target"]).copy()

    # Bitcoin keeps Twitter features; ETH/DOGE are explicitly price-only
    if coin == "Bitcoin":
        coin_feat_cols = feat_cols.copy()
    else:
        coin_feat_cols = [c for c in feat_cols if c not in twitter_feature_cols]

    # Replace inf before row filtering / imputation
    g[coin_feat_cols] = g[coin_feat_cols].replace([np.inf, -np.inf], np.nan)

    # Drop rows missing >20% of the selected feature set for that coin
    g = g.dropna(subset=coin_feat_cols, thresh=int(0.8 * len(coin_feat_cols)))

    # Remaining NaN -> fill with column median
    g[coin_feat_cols] = g[coin_feat_cols].fillna(g[coin_feat_cols].median())

    fname = f"features_{coin.lower()}.csv"
    g[["coin", "date"] + coin_feat_cols + ["target"]].to_csv(fname, index=False)

    feature_cols_by_coin[coin] = coin_feat_cols
    pd.DataFrame({"feature": coin_feat_cols}).to_csv(f"feature_cols_{coin.lower()}.csv", index=False)

    twitter_mode = "price + Twitter" if coin == "Bitcoin" else "price only"
    print(
        f"Saved {fname}  shape={g.shape}  target balance: {g['target'].mean():.1%} up"
        f"  | mode: {twitter_mode}  | features: {len(coin_feat_cols)}"
    )

print("\nSaved feature lists:")
for coin, cols in feature_cols_by_coin.items():
    print(f"- feature_cols_{coin.lower()}.csv ({len(cols)} features)")

print("\n✅ All feature files saved.")


Saved features_bitcoin.csv  shape=(4037, 71)  target balance: 48.8% up  | mode: price + Twitter  | features: 60
Saved features_ethereum.csv  shape=(1975, 71)  target balance: 50.9% up  | mode: price only  | features: 51
Saved features_dogecoin.csv  shape=(1525, 71)  target balance: 48.5% up  | mode: price only  | features: 51

Saved feature lists:
- feature_cols_bitcoin.csv (60 features)
- feature_cols_ethereum.csv (51 features)
- feature_cols_dogecoin.csv (51 features)

✅ All feature files saved.
